In [ ]:
import pandas as pd
import numpy as np
import scanpy as sc
import decoupler as dc
from pypath.inputs import msigdb
import seaborn as sns

import matplotlib.pyplot as plt
from matplotlib.pyplot import rc_context
from matplotlib.backends.backend_pdf import PdfPages

import yaml
from pathlib import Path

# for GO enrichment analysis
import tempfile
import tqdm.auto as tqdm
import gseapy as gp
from gseapy.plot import barplot, dotplot
from gseapy import get_library_name
from gseapy import gseaplot
import gseapy.plot as gsplot

#import requests
#import json
#import io

import itertools

# Data import and preparation

In [ ]:
adata = sc.read("./annotated-aging-soupxed.h5ad", var_names="gene_symbols")

adata_PT1 = adata[adata.obs['celltype'] == 'PT-1'].copy()
adata_PT2 = adata[adata.obs['celltype'] == 'PT-2'].copy()
adata_EC2 = adata[adata.obs['celltype'] == 'EC-2'].copy()
adata_TAL1 = adata[adata.obs['celltype'] == 'TAL-1'].copy()
adata_IMM = adata[adata.obs['celltype'] == 'IMM'].copy()

# Gloms
adata_PODO = adata[adata.obs['celltype'] == 'PODO'].copy()
adata_vSMC = adata[adata.obs['celltype'] == 'vSMC/MC'].copy()
adata_PEC = adata[adata.obs['celltype'] == 'PECs?'].copy()
adata_EC1 = adata[adata.obs['celltype'] == 'EC-1(gEC)'].copy()

In [ ]:
unique_combinations = adata.obs[['sample', 'orig.ident']].drop_duplicates()
print(unique_combinations)

In [ ]:
output_dir = "/data/projects/manuela/aging/scRNA_aging-mouse/deg_tables_vsSG20/"

# Analysis by Samples by Celltypes

In [ ]:
adata_PT1.uns['log1p']['base'] = None
adata_PT2.uns['log1p']['base'] = None
adata_EC2.uns['log1p']['base'] = None
adata_TAL1.uns['log1p']['base'] = None
adata_IMM.uns['log1p']['base'] = None

adata.uns['log1p']['base'] = None
adata_PODO.uns['log1p']['base'] = None
adata_vSMC.uns['log1p']['base'] = None
adata_PEC.uns['log1p']['base'] = None
adata_EC1.uns['log1p']['base'] = None

sc.tl.rank_genes_groups(adata, 'sample', method='t-test', reference = 'age')

sc.tl.rank_genes_groups(adata_PT1, 'sample', method='t-test', reference = 'age')
sc.tl.rank_genes_groups(adata_PT2, 'sample', method='t-test', reference = 'age')
sc.tl.rank_genes_groups(adata_EC2, 'sample', method='t-test', reference = 'age')
sc.tl.rank_genes_groups(adata_TAL1, 'sample', method='t-test', reference = 'age')
sc.tl.rank_genes_groups(adata_IMM, 'sample', method='t-test', reference = 'age')

# Gloms
sc.tl.rank_genes_groups(adata_PODO, 'sample', method='t-test', reference = 'age')
sc.tl.rank_genes_groups(adata_vSMC, 'sample', method='t-test', reference = 'age')
sc.tl.rank_genes_groups(adata_PEC, 'sample', method='t-test', reference = 'age')
sc.tl.rank_genes_groups(adata_EC1, 'sample', method='t-test', reference = 'age')

## DGE Analysis

### Full adata

In [ ]:
p = 0.05
log2fc = 0 

deg = sc.get.rank_genes_groups_df(adata, group= None)
## If log2FoldChange > 0 and pvalue < 0.05, set as "UP"
deg.loc[(deg.logfoldchanges > 0) & (deg.pvals_adj < 0.05), 'sign'] = "upregulated"
## If log2FoldChange < 0 and pvalue < 0.05, set as "DOWN"
deg.loc[(deg.logfoldchanges < 0) & (deg.pvals_adj < 0.05), 'sign'] = "downregulated"

# significant genes -> filtered by pval and log2fc
deg_sig = deg[(deg['pvals_adj'] < p) & (deg['logfoldchanges'].abs() > log2fc)]

# Filter by 'sign' and sort
deg_sig_up = deg_sig[deg_sig['sign'] == 'upregulated'].sort_values(by='logfoldchanges', ascending=False)
deg_sig_down = deg_sig[deg_sig['sign'] == 'downregulated'].sort_values(by='logfoldchanges', ascending=True)

# Split by 'group'
deg_sig_up_ctrl = deg_sig_up[deg_sig_up['group'] == 'ctrl']
deg_sig_up_sAct = deg_sig_up[deg_sig_up['group'] == 'sAct']
deg_sig_up_DR = deg_sig_up[deg_sig_up['group'] == 'DR']
deg_sig_up_combi = deg_sig_up[deg_sig_up['group'] == 'combi']

deg_sig_down_ctrl = deg_sig_down[deg_sig_down['group'] == 'ctrl']
deg_sig_down_sAct = deg_sig_down[deg_sig_down['group'] == 'sAct']
deg_sig_down_DR = deg_sig_down[deg_sig_down['group'] == 'DR']
deg_sig_down_combi = deg_sig_down[deg_sig_down['group'] == 'combi']

In [ ]:
def save_to_csv(df, filename):
    df.to_csv(output_dir + filename, index=False)

In [ ]:
# Save up genes
save_to_csv(deg_sig_up_ctrl, "up_ctrl_vs_aging.csv")
save_to_csv(deg_sig_up_sAct, "up_sAct_vs_aging.csv")
save_to_csv(deg_sig_up_DR, "up_DR_vs_aging.csv")
save_to_csv(deg_sig_up_combi, "up_combi_vs_aging.csv")

# Save down genes
save_to_csv(deg_sig_down_ctrl, "down_ctrl_vs_aging.csv")
save_to_csv(deg_sig_down_sAct, "down_sAct_vs_aging.csv")
save_to_csv(deg_sig_down_DR, "down_DR_vs_aging.csv")
save_to_csv(deg_sig_down_combi, "down_combi_vs_aging.csv")

### PODO significant DEGs

In [ ]:
sc.pl.rank_genes_groups(adata_PODO, n_genes = 20)

In [ ]:
p = 0.05
log2fc = 0 

deg_podo = sc.get.rank_genes_groups_df(adata_PODO, group= None)
## If log2FoldChange > 0 and pvalue < 0.05, set as "UP"
deg_podo.loc[(deg_podo.logfoldchanges > 0) & (deg_podo.pvals_adj < 0.05), 'sign'] = "upregulated"
## If log2FoldChange < 0 and pvalue < 0.05, set as "DOWN"
deg_podo.loc[(deg_podo.logfoldchanges < 0) & (deg_podo.pvals_adj < 0.05), 'sign'] = "downregulated"

# filtered only by pval
deg_podo_fi = deg_podo[deg_podo['pvals_adj'] < p]

# significant genes -> filtered by pval and log2fc
deg_podo_sig = deg_podo[(deg_podo['pvals_adj'] < p) & (deg_podo['logfoldchanges'].abs() > log2fc)]

deg_podo_sig_sg18 = deg_podo_sig[deg_podo_sig['group']=='ctrl']
deg_podo_sig_sg24 = deg_podo_sig[deg_podo_sig['group']=='sAct']
deg_podo_sig_sg26 = deg_podo_sig[deg_podo_sig['group']=='DR']
deg_podo_sig_sg28 = deg_podo_sig[deg_podo_sig['group']=='combi']

print(deg_podo.shape)
print(deg_podo_fi.shape)
print(deg_podo_sig.shape)
deg_podo_sig_sg18

#### Heatmap lfc

In [ ]:
# Pivot the DataFrame to create a matrix suitable for a heatmap
heatmap_matrix = deg_podo_sig.pivot(index='group', columns='names', values='logfoldchanges')

#figure
plt.figure(figsize=(16,6))
plt.subplots_adjust(left=0.15, bottom=0.25) 
ax = sns.heatmap(heatmap_matrix, cmap='coolwarm', annot=False, fmt='.2f')
ax.set_title("Logfoldchanges of DEGs in PODO")
plt.show()

####  Heatmap Gene Expression 

In [ ]:
#plt.title('Gene expression of last 50 entries sorted by lfcs')
sc.pl.heatmap(adata_PODO, var_names=deg_podo_sig['names'][-50:], groupby='sample', cmap='Blues_r', dendrogram=True, vmin=0, vmax=4)

#plt.title('Gene expression of first 50 entries sorted by lfcs')
sc.pl.heatmap(adata_PODO, var_names=deg_podo_sig['names'][:50], groupby='sample', cmap='Blues_r', dendrogram=True, vmin=0, vmax=4)

#### Dotplot 

In [ ]:
len(list(set(deg_podo_sig['names'])))

In [ ]:
sc.pl.DotPlot(adata_PODO, use_raw=True, var_names = list(set(deg_podo_sig['names'])), groupby='sample', standard_scale="var", title='PODO DEGs', cmap='BuGn', figsize=(300, 5)).savefig('./figures/sig_deg_podo_dotplot.pdf' ,format='pdf')

#### Vulcanoplot

In [ ]:
# Create a column to classify genes as upregulated or downregulated
deg_podo['Regulation'] = 'Not Significant'  # Initialize as not significant
#deg_podo.loc[(deg_podo['logfoldchanges'] > 0) & (deg_podo['pvals_adj'] < 0.05), 'Regulation'] = 'Upregulated'
#deg_podo.loc[(deg_podo['logfoldchanges'] < 0) & (deg_podo['pvals_adj'] < 0.05), 'Regulation'] = 'Downregulated'
deg_podo.loc[(deg_podo['pvals_adj'] < 0.05) & (deg_podo['group'] == 'ctrl'), 'Regulation'] = 'ctrl'
deg_podo.loc[(deg_podo['pvals_adj'] < 0.05) & (deg_podo['group'] == 'sAct'), 'Regulation'] = 'sAct'
deg_podo.loc[(deg_podo['pvals_adj'] < 0.05) & (deg_podo['group'] == 'DR'), 'Regulation'] = 'DR'
deg_podo.loc[(deg_podo['pvals_adj'] < 0.05) & (deg_podo['group'] == 'combi'), 'Regulation'] = 'combi'

# Define colors for significant genes
colors = {'ctrl': 'blue','sAct': 'green','DR': 'purple','combi': 'orange', 'Not Significant': 'gray'}

plt.figure(figsize=(20, 10))
for regulation, color in colors.items():
    subset_df = deg_podo[deg_podo['Regulation'] == regulation]
    plt.scatter(subset_df['logfoldchanges'], -np.log10(subset_df['pvals_adj']), color=color, alpha=0.3, label=regulation)
   
    
# Annotate all significant genes (adjusted P-value < 0.05) with their gene names
significant_genes = deg_podo[deg_podo['pvals_adj'] < 0.05]
#for index, row in significant_genes.iterrows():
#    plt.annotate(row['names'], (row['logfoldchanges'], -np.log10(row['pvals_adj'])), color='black', fontsize=8)
    
plt.xlabel('Log2(Fold Change)')
plt.xlabel('Log Fold Change')
plt.ylabel('-log10(P-Value)')
plt.title('Volcano Plot for Podo')
plt.axvline(x=-1*log2fc, linestyle='--', color='gray')
plt.axhline(-np.log10(0.05), color='red', linestyle='--', label='Significance Threshold (Adj. p-value < 0.05)')
plt.legend()
plt.show()

### vSMC significant DEGs

In [ ]:
sc.pl.rank_genes_groups(adata_vSMC, n_genes = 20)

In [ ]:
p = 0.05
log2fc = 0 

deg_vsmc = sc.get.rank_genes_groups_df(adata_vSMC, group= None)
## If log2FoldChange > 0 and pvalue < 0.05, set as "UP"
deg_vsmc.loc[(deg_vsmc.logfoldchanges > 0) & (deg_vsmc.pvals_adj < 0.05), 'sign'] = "upregulated"
## If log2FoldChange < 0 and pvalue < 0.05, set as "DOWN"
deg_vsmc.loc[(deg_vsmc.logfoldchanges < 0) & (deg_vsmc.pvals_adj < 0.05), 'sign'] = "downregulated"

# filtered only by pval
deg_vsmc_fi = deg_vsmc[deg_vsmc['pvals_adj'] < p]

# significant genes -> filtered by pval and log2fc
deg_vsmc_sig = deg_vsmc[(deg_vsmc['pvals_adj'] < p) & (deg_vsmc['logfoldchanges'].abs() > log2fc)]

deg_vsmc_sig_sg18 = deg_vsmc_sig[deg_vsmc_sig['group']=='ctrl']
deg_vsmc_sig_sg24 = deg_vsmc_sig[deg_vsmc_sig['group']=='sAct']
deg_vsmc_sig_sg26 = deg_vsmc_sig[deg_vsmc_sig['group']=='DR']
deg_vsmc_sig_sg28 = deg_vsmc_sig[deg_vsmc_sig['group']=='combi']

print(deg_vsmc.shape)
print(deg_vsmc_fi.shape)
print(deg_vsmc_sig.shape)

#### Heatmap lfc

In [ ]:
# Pivot the DataFrame to create a matrix suitable for a heatmap
heatmap_matrix = deg_vsmc_sig.pivot(index='group', columns='names', values='logfoldchanges')

#figure
plt.figure(figsize=(16,6))
plt.subplots_adjust(left=0.15, bottom=0.25) 
ax = sns.heatmap(heatmap_matrix, cmap='coolwarm', annot=False, fmt='.2f')
ax.set_title("Logfoldchanges of DEGs in vSMC/MC")

plt.show()

#### Vulcanoplot

In [ ]:
# Create a column to classify genes as upregulated or downregulated
deg_vsmc['Regulation'] = 'Not Significant'  # Initialize as not significant
#deg_vsmc.loc[(deg_vsmc['logfoldchanges'] > 0) & (deg_vsmc['pvals_adj'] < 0.05), 'Regulation'] = 'Upregulated'
#deg_vsmc.loc[(deg_vsmc['logfoldchanges'] < 0) & (deg_vsmc['pvals_adj'] < 0.05), 'Regulation'] = 'Downregulated'
deg_vsmc.loc[(deg_vsmc['pvals_adj'] < 0.05) & (deg_vsmc['group'] == 'ctrl'), 'Regulation'] = 'ctrl'
deg_vsmc.loc[(deg_vsmc['pvals_adj'] < 0.05) & (deg_vsmc['group'] == 'sAct'), 'Regulation'] = 'sAct'
deg_vsmc.loc[(deg_vsmc['pvals_adj'] < 0.05) & (deg_vsmc['group'] == 'DR'), 'Regulation'] = 'DR'
deg_vsmc.loc[(deg_vsmc['pvals_adj'] < 0.05) & (deg_vsmc['group'] == 'combi'), 'Regulation'] = 'combi'

# Define colors for significant genes
colors = {'ctrl': 'blue','sAct': 'green','DR': 'purple','combi': 'orange', 'Not Significant': 'gray'}

plt.figure(figsize=(20, 10))
for regulation, color in colors.items():
    subset_df = deg_vsmc[deg_vsmc['Regulation'] == regulation]
    plt.scatter(subset_df['logfoldchanges'], -np.log10(subset_df['pvals_adj']), color=color, alpha=0.3, label=regulation)
   
    
# Annotate all significant genes (adjusted P-value < 0.05) with their gene names
significant_genes = deg_vsmc[deg_vsmc['pvals_adj'] < 0.05]
#for index, row in significant_genes.iterrows():
#    plt.annotate(row['names'], (row['logfoldchanges'], -np.log10(row['pvals_adj'])), color='black', fontsize=8)
    
plt.xlabel('Log2(Fold Change)')
plt.xlabel('Log Fold Change')
plt.ylabel('-log10(P-Value)')
plt.title('Volcano Plot for vSMC')
plt.axvline(x=-1*log2fc, linestyle='--', color='gray')
plt.axhline(-np.log10(0.05), color='red', linestyle='--', label='Significance Threshold (Adj. p-value < 0.05)')
plt.legend()
plt.show()

### PEC significant DEGs

In [ ]:
sc.pl.rank_genes_groups(adata_PEC, n_genes = 20)

In [ ]:
p = 0.05
log2fc = 0 

deg_pec = sc.get.rank_genes_groups_df(adata_PEC, group= None)
## If log2FoldChange > 0 and pvalue < 0.05, set as "UP"
deg_pec.loc[(deg_pec.logfoldchanges > 0) & (deg_pec.pvals_adj < 0.05), 'sign'] = "upregulated"
## If log2FoldChange < 0 and pvalue < 0.05, set as "DOWN"
deg_pec.loc[(deg_pec.logfoldchanges < 0) & (deg_pec.pvals_adj < 0.05), 'sign'] = "downregulated"

# filtered only by pval
deg_pec_fi = deg_pec[deg_pec['pvals_adj'] < p]

# significant genes -> filtered by pval and log2fc
deg_pec_sig = deg_pec[(deg_pec['pvals_adj'] < p) & (deg_pec['logfoldchanges'].abs() > log2fc)]

deg_pec_sig_sg18 = deg_pec_sig[deg_pec_sig['group']=='ctrl']
deg_pec_sig_sg24 = deg_pec_sig[deg_pec_sig['group']=='sAct']
deg_pec_sig_sg26 = deg_pec_sig[deg_pec_sig['group']=='DR']
deg_pec_sig_sg28 = deg_pec_sig[deg_pec_sig['group']=='combi']

print(deg_pec.shape)
print(deg_pec_fi.shape)
print(deg_pec_sig.shape)

#### Heatmap lfc

In [ ]:
# Pivot the DataFrame to create a matrix suitable for a heatmap
heatmap_matrix = deg_pec_sig.pivot(index='group', columns='names', values='logfoldchanges')

#figure
plt.figure(figsize=(16,6))
plt.subplots_adjust(left=0.15, bottom=0.25) 
ax = sns.heatmap(heatmap_matrix, cmap='coolwarm', annot=False, fmt='.2f')
ax.set_title("Logfoldchanges of DEGs in PEC")

plt.show()

#### Vulcanoplot

In [ ]:
# Create a column to classify genes as upregulated or downregulated
deg_pec['Regulation'] = 'Not Significant'  # Initialize as not significant
#deg_pec.loc[(deg_pec['logfoldchanges'] > 0) & (deg_pec['pvals_adj'] < 0.05), 'Regulation'] = 'Upregulated'
#deg_pec.loc[(deg_pec['logfoldchanges'] < 0) & (deg_pec['pvals_adj'] < 0.05), 'Regulation'] = 'Downregulated'
deg_pec.loc[(deg_pec['pvals_adj'] < 0.05) & (deg_pec['group'] == 'ctrl'), 'Regulation'] = 'ctrl'
deg_pec.loc[(deg_pec['pvals_adj'] < 0.05) & (deg_pec['group'] == 'sAct'), 'Regulation'] = 'sAct'
deg_pec.loc[(deg_pec['pvals_adj'] < 0.05) & (deg_pec['group'] == 'DR'), 'Regulation'] = 'DR'
deg_pec.loc[(deg_pec['pvals_adj'] < 0.05) & (deg_pec['group'] == 'combi'), 'Regulation'] = 'combi'

# Define colors for significant genes
colors = {'ctrl': 'blue','sAct': 'green','DR': 'purple','combi': 'orange', 'Not Significant': 'gray'}

plt.figure(figsize=(20, 10))
for regulation, color in colors.items():
    subset_df = deg_pec[deg_pec['Regulation'] == regulation]
    plt.scatter(subset_df['logfoldchanges'], -np.log10(subset_df['pvals_adj']), color=color, alpha=0.3, label=regulation)
   
    
# Annotate all significant genes (adjusted P-value < 0.05) with their gene names
significant_genes = deg_pec[deg_pec['pvals_adj'] < 0.05]
#for index, row in significant_genes.iterrows():
#    plt.annotate(row['names'], (row['logfoldchanges'], -np.log10(row['pvals_adj'])), color='black', fontsize=8)
    
plt.xlabel('Log2(Fold Change)')
plt.xlabel('Log Fold Change')
plt.ylabel('-log10(P-Value)')
plt.title('Volcano Plot for PEC')
plt.axvline(x=-1*log2fc, linestyle='--', color='gray')
plt.axhline(-np.log10(0.05), color='red', linestyle='--', label='Significance Threshold (Adj. p-value < 0.05)')
plt.legend()
plt.show()

### EC1 (gEC) significant DEGs

In [ ]:
sc.pl.rank_genes_groups(adata_EC1, n_genes = 20)

In [ ]:
p = 0.05
log2fc = 0 

deg_ec1 = sc.get.rank_genes_groups_df(adata_EC1, group= None)
## If log2FoldChange > 0 and pvalue < 0.05, set as "UP"
deg_ec1.loc[(deg_ec1.logfoldchanges > 0) & (deg_ec1.pvals_adj < 0.05), 'sign'] = "upregulated"
## If log2FoldChange < 0 and pvalue < 0.05, set as "DOWN"
deg_ec1.loc[(deg_ec1.logfoldchanges < 0) & (deg_ec1.pvals_adj < 0.05), 'sign'] = "downregulated"

# filtered only by pval
deg_ec1_fi = deg_ec1[deg_ec1['pvals_adj'] < p]

# significant genes -> filtered by pval and log2fc
deg_ec1_sig = deg_ec1[(deg_ec1['pvals_adj'] < p) & (deg_ec1['logfoldchanges'].abs() > log2fc)]

deg_ec1_sig_sg18 = deg_ec1_sig[deg_ec1_sig['group']=='ctrl']
deg_ec1_sig_sg24 = deg_ec1_sig[deg_ec1_sig['group']=='sAct']
deg_ec1_sig_sg26 = deg_ec1_sig[deg_ec1_sig['group']=='DR']
deg_ec1_sig_sg28 = deg_ec1_sig[deg_ec1_sig['group']=='combi']

print(deg_ec1.shape)
print(deg_ec1_fi.shape)
print(deg_ec1_sig.shape)

#### Heatmap lfc

In [ ]:
# Pivot the DataFrame to create a matrix suitable for a heatmap
heatmap_matrix = deg_ec1_sig.pivot(index='group', columns='names', values='logfoldchanges')

#figure
plt.figure(figsize=(16,6))
plt.subplots_adjust(left=0.15, bottom=0.25) 
ax = sns.heatmap(heatmap_matrix, cmap='coolwarm', annot=False, fmt='.2f')
ax.set_title("Logfoldchanges of DEGs in EC1")

plt.show()

#### Vulcanoplot

In [ ]:
# Create a column to classify genes as upregulated or downregulated
deg_ec1['Regulation'] = 'Not Significant'  # Initialize as not significant
#deg_ec1.loc[(deg_ec1['logfoldchanges'] > 0) & (deg_ec1['pvals_adj'] < 0.05), 'Regulation'] = 'Upregulated'
#deg_ec1.loc[(deg_ec1['logfoldchanges'] < 0) & (deg_ec1['pvals_adj'] < 0.05), 'Regulation'] = 'Downregulated'
deg_ec1.loc[(deg_ec1['pvals_adj'] < 0.05) & (deg_ec1['group'] == 'ctrl'), 'Regulation'] = 'ctrl'
deg_ec1.loc[(deg_ec1['pvals_adj'] < 0.05) & (deg_ec1['group'] == 'sAct'), 'Regulation'] = 'sAct'
deg_ec1.loc[(deg_ec1['pvals_adj'] < 0.05) & (deg_ec1['group'] == 'DR'), 'Regulation'] = 'DR'
deg_ec1.loc[(deg_ec1['pvals_adj'] < 0.05) & (deg_ec1['group'] == 'combi'), 'Regulation'] = 'combi'

# Define colors for significant genes
colors = {'ctrl': 'blue','sAct': 'green','DR': 'purple','combi': 'orange', 'Not Significant': 'gray'}

plt.figure(figsize=(20, 10))
for regulation, color in colors.items():
    subset_df = deg_ec1[deg_ec1['Regulation'] == regulation]
    plt.scatter(subset_df['logfoldchanges'], -np.log10(subset_df['pvals_adj']), color=color, alpha=0.3, label=regulation)
   
    
# Annotate all significant genes (adjusted P-value < 0.05) with their gene names
significant_genes = deg_ec1[deg_ec1['pvals_adj'] < 0.05]
#for index, row in significant_genes.iterrows():
 #   plt.annotate(row['names'], (row['logfoldchanges'], -np.log10(row['pvals_adj'])), color='black', fontsize=8)
    
plt.xlabel('Log2(Fold Change)')
plt.xlabel('Log Fold Change')
plt.ylabel('-log10(P-Value)')
plt.title('Volcano Plot for EC1(gEC)')
plt.axvline(x=-1*log2fc, linestyle='--', color='gray')
plt.axhline(-np.log10(0.05), color='red', linestyle='--', label='Significance Threshold (Adj. p-value < 0.05)')
plt.legend()
plt.show()

### PT1 significant DEGs

In [ ]:
sc.pl.rank_genes_groups(adata_PT1, n_genes = 20)

In [ ]:
p = 0.05
log2fc = 0 

deg_pt1 = sc.get.rank_genes_groups_df(adata_PT1, group= None)
## If log2FoldChange > 0 and pvalue < 0.05, set as "UP"
deg_pt1.loc[(deg_pt1.logfoldchanges > 0) & (deg_pt1.pvals_adj < 0.05), 'sign'] = "upregulated"
## If log2FoldChange < 0 and pvalue < 0.05, set as "DOWN"
deg_pt1.loc[(deg_pt1.logfoldchanges < 0) & (deg_pt1.pvals_adj < 0.05), 'sign'] = "downregulated"

# filtered only by pval
deg_pt1_fi = deg_pt1[deg_pt1['pvals_adj'] < p]

# significant genes -> filtered by pval and log2fc
deg_pt1_sig = deg_pt1[(deg_pt1['pvals_adj'] < p) & (deg_pt1['logfoldchanges'].abs() > log2fc)]

deg_pt1_sig_sg18 = deg_pt1_sig[deg_pt1_sig['group']=='ctrl']
deg_pt1_sig_sg24 = deg_pt1_sig[deg_pt1_sig['group']=='sAct']
deg_pt1_sig_sg26 = deg_pt1_sig[deg_pt1_sig['group']=='DR']
deg_pt1_sig_sg28 = deg_pt1_sig[deg_pt1_sig['group']=='combi']

print(deg_pt1.shape)
print(deg_pt1_fi.shape)
print(deg_pt1_sig.shape)


#### Heatmap lfc

In [ ]:
# Pivot the DataFrame to create a matrix suitable for a heatmap
heatmap_matrix = deg_pt1_sig.pivot(index='group', columns='names', values='logfoldchanges')

#figure
plt.figure(figsize=(16,6))
plt.subplots_adjust(left=0.15, bottom=0.25) 
ax = sns.heatmap(heatmap_matrix, cmap='coolwarm', annot=False, fmt='.2f')
ax.set_title("Logfoldchanges of DEGs in PT1")

plt.show()

#### Vulcanoplot

In [ ]:
# Create a column to classify genes as upregulated or downregulated
deg_pt1['Regulation'] = 'Not Significant'  # Initialize as not significant
#deg_pt1.loc[(deg_pt1['logfoldchanges'] > 0) & (deg_pt1['pvals_adj'] < 0.05), 'Regulation'] = 'Upregulated'
#deg_pt1.loc[(deg_pt1['logfoldchanges'] < 0) & (deg_pt1['pvals_adj'] < 0.05), 'Regulation'] = 'Downregulated'
deg_pt1.loc[(deg_pt1['pvals_adj'] < 0.05) & (deg_pt1['group'] == 'ctrl'), 'Regulation'] = 'ctrl'
deg_pt1.loc[(deg_pt1['pvals_adj'] < 0.05) & (deg_pt1['group'] == 'sAct'), 'Regulation'] = 'sAct'
deg_pt1.loc[(deg_pt1['pvals_adj'] < 0.05) & (deg_pt1['group'] == 'DR'), 'Regulation'] = 'DR'
deg_pt1.loc[(deg_pt1['pvals_adj'] < 0.05) & (deg_pt1['group'] == 'combi'), 'Regulation'] = 'combi'

# Define colors for significant genes
colors = {'ctrl': 'blue','sAct': 'green','DR': 'purple','combi': 'orange', 'Not Significant': 'gray'}

plt.figure(figsize=(20, 10))
for regulation, color in colors.items():
    subset_df = deg_pt1[deg_pt1['Regulation'] == regulation]
    plt.scatter(subset_df['logfoldchanges'], -np.log10(subset_df['pvals_adj']), color=color, alpha=0.3, label=regulation)
   
    
# Annotate all significant genes (adjusted P-value < 0.05) with their gene names
significant_genes = deg_pt1[deg_pt1['pvals_adj'] < 0.05]
#for index, row in significant_genes.iterrows():
#    plt.annotate(row['names'], (row['logfoldchanges'], -np.log10(row['pvals_adj'])), color='black', fontsize=8)
    
plt.xlabel('Log2(Fold Change)')
plt.xlabel('Log Fold Change')
plt.ylabel('-log10(P-Value)')
plt.title('Volcano Plot for PT1')
plt.axvline(x=-1*log2fc, linestyle='--', color='gray')
plt.axhline(-np.log10(0.05), color='red', linestyle='--', label='Significance Threshold (Adj. p-value < 0.05)')
plt.legend()
plt.show()

#### Dotplot 

In [ ]:
sc.pl.DotPlot(adata_PT1, use_raw=True, var_names = list(set(deg_pt1_sig['names'])), groupby='sample', standard_scale="var", title='PT1 DEGs', cmap='BuGn', figsize=(300, 5)).savefig('./figures/sig_deg_pt1_dotplot.pdf' ,format='pdf')

### PT2 significant DEGs

In [ ]:
p = 0.05
log2fc = 0 

deg_pt2 = sc.get.rank_genes_groups_df(adata_PT2, group= None)
## If log2FoldChange > 0 and pvalue < 0.05, set as "UP"
deg_pt2.loc[(deg_pt2.logfoldchanges > 0) & (deg_pt2.pvals_adj < 0.05), 'sign'] = "upregulated"
## If log2FoldChange < 0 and pvalue < 0.05, set as "DOWN"
deg_pt2.loc[(deg_pt2.logfoldchanges < 0) & (deg_pt2.pvals_adj < 0.05), 'sign'] = "downregulated"

# filtered only by pval
deg_pt2_fi = deg_pt2[deg_pt2['pvals_adj'] < p]

# significant genes -> filtered by pval and log2fc
deg_pt2_sig = deg_pt2[(deg_pt2['pvals_adj'] < p) & (deg_pt2['logfoldchanges'].abs() > log2fc)]

deg_pt2_sig_sg18 = deg_pt2_sig[deg_pt2_sig['group']=='ctrl']
deg_pt2_sig_sg24 = deg_pt2_sig[deg_pt2_sig['group']=='sAct']
deg_pt2_sig_sg26 = deg_pt2_sig[deg_pt2_sig['group']=='DR']
deg_pt2_sig_sg28 = deg_pt2_sig[deg_pt2_sig['group']=='combi']

print(deg_pt2.shape)
print(deg_pt2_fi.shape)
print(deg_pt2_sig.shape)


#### Heatmap lfc

In [ ]:
# Pivot the DataFrame to create a matrix suitable for a heatmap
heatmap_matrix = deg_pt2_sig.pivot(index='group', columns='names', values='logfoldchanges')

#figure
plt.figure(figsize=(16,6))
plt.subplots_adjust(left=0.15, bottom=0.25) 
ax = sns.heatmap(heatmap_matrix, cmap='coolwarm', annot=False, fmt='.2f')
ax.set_title("Logfoldchanges of DEGs in PT2")

plt.show()

#### Vulcanoplot

In [ ]:
# Create a column to classify genes as upregulated or downregulated
deg_pt2['Regulation'] = 'Not Significant'  # Initialize as not significant
#deg_pt2.loc[(deg_pt2['logfoldchanges'] > 0) & (deg_pt2['pvals_adj'] < 0.05), 'Regulation'] = 'Upregulated'
#deg_pt2.loc[(deg_pt2['logfoldchanges'] < 0) & (deg_pt2['pvals_adj'] < 0.05), 'Regulation'] = 'Downregulated'
deg_pt2.loc[(deg_pt2['pvals_adj'] < 0.05) & (deg_pt2['group'] == 'ctrl'), 'Regulation'] = 'ctrl'
deg_pt2.loc[(deg_pt2['pvals_adj'] < 0.05) & (deg_pt2['group'] == 'sAct'), 'Regulation'] = 'sAct'
deg_pt2.loc[(deg_pt2['pvals_adj'] < 0.05) & (deg_pt2['group'] == 'DR'), 'Regulation'] = 'DR'
deg_pt2.loc[(deg_pt2['pvals_adj'] < 0.05) & (deg_pt2['group'] == 'combi'), 'Regulation'] = 'combi'

# Define colors for significant genes
colors = {'ctrl': 'blue','sAct': 'green','DR': 'purple','combi': 'orange', 'Not Significant': 'gray'}

plt.figure(figsize=(20, 10))
for regulation, color in colors.items():
    subset_df = deg_pt2[deg_pt2['Regulation'] == regulation]
    plt.scatter(subset_df['logfoldchanges'], -np.log10(subset_df['pvals_adj']), color=color, alpha=0.3, label=regulation)
   
    
# Annotate all significant genes (adjusted P-value < 0.05) with their gene names
significant_genes = deg_pt2[deg_pt2['pvals_adj'] < 0.05]
#for index, row in significant_genes.iterrows():
#    plt.annotate(row['names'], (row['logfoldchanges'], -np.log10(row['pvals_adj'])), color='black', fontsize=8)
    
plt.xlabel('Log2(Fold Change)')
plt.xlabel('Log Fold Change')
plt.ylabel('-log10(P-Value)')
plt.title('Volcano Plot for PT-2')
plt.axvline(x=-1*log2fc, linestyle='--', color='gray')
plt.axhline(-np.log10(0.05), color='red', linestyle='--', label='Significance Threshold (Adj. p-value < 0.05)')
plt.legend()
plt.show()

#### Dotplot

In [ ]:
sc.pl.DotPlot(adata_PT2, use_raw=True, var_names = list(set(deg_pt2_sig['names'])), groupby='sample', standard_scale="var", title='PT2 DEGs', cmap='BuGn', figsize=(300, 5)).savefig('./figures/sig_deg_pt2_dotplot.pdf',format='pdf')

### EC2 significant DEGs

In [ ]:
p = 0.05
log2fc = 0 

deg_ec2 = sc.get.rank_genes_groups_df(adata_EC2, group= None)
## If log2FoldChange > 0 and pvalue < 0.05, set as "UP"
deg_ec2.loc[(deg_ec2.logfoldchanges > 0) & (deg_ec2.pvals_adj < 0.05), 'sign'] = "upregulated"
## If log2FoldChange < 0 and pvalue < 0.05, set as "DOWN"
deg_ec2.loc[(deg_ec2.logfoldchanges < 0) & (deg_ec2.pvals_adj < 0.05), 'sign'] = "downregulated"

# filtered only by pval
deg_ec2_fi = deg_ec2[deg_ec2['pvals_adj'] < p]

# significant genes -> filtered by pval and log2fc
deg_ec2_sig = deg_ec2[(deg_ec2['pvals_adj'] < p) & (deg_ec2['logfoldchanges'].abs() > log2fc)]

deg_ec2_sig_sg18 = deg_ec2_sig[deg_ec2_sig['group']=='ctrl']
deg_ec2_sig_sg24 = deg_ec2_sig[deg_ec2_sig['group']=='sAct']
deg_ec2_sig_sg26 = deg_ec2_sig[deg_ec2_sig['group']=='DR']
deg_ec2_sig_sg28 = deg_ec2_sig[deg_ec2_sig['group']=='combi']

print(deg_ec2.shape)
print(deg_ec2_fi.shape)
print(deg_ec2_sig.shape)

#### Heatmap lfc

In [ ]:
# Pivot the DataFrame to create a matrix suitable for a heatmap
heatmap_matrix = deg_ec2_sig.pivot(index='group', columns='names', values='logfoldchanges')

#figure
plt.figure(figsize=(16,4))
ax = sns.heatmap(heatmap_matrix, cmap='coolwarm', annot=False, fmt='.2f')
ax.set_title("Logfoldchanges of DEGs in EC2")
plt.show()

#### Vulcanoplot

#### Dotplot 

In [ ]:
sc.pl.DotPlot(adata_EC2, use_raw=True, var_names = list(set(deg_ec2_sig['names'])), groupby='sample', standard_scale="var", title='EC2 DEGs', cmap='BuGn', figsize=(300, 5)).savefig('./figures/sig_deg_ec2_dotplot.pdf')

### TAL1 significant DEGs

In [ ]:
p = 0.05
log2fc = 0 

deg_tal1 = sc.get.rank_genes_groups_df(adata_TAL1, group= None)
## If log2FoldChange > 0 and pvalue < 0.05, set as "UP"
deg_tal1.loc[(deg_tal1.logfoldchanges > 0) & (deg_tal1.pvals_adj < 0.05), 'sign'] = "upregulated"
## If log2FoldChange < 0 and pvalue < 0.05, set as "DOWN"
deg_tal1.loc[(deg_tal1.logfoldchanges < 0) & (deg_tal1.pvals_adj < 0.05), 'sign'] = "downregulated"

# filtered only by pval
deg_tal1_fi = deg_tal1[deg_tal1['pvals_adj'] < p]

# significant genes -> filtered by pval and log2fc
deg_tal1_sig = deg_tal1[(deg_tal1['pvals_adj'] < p) & (deg_tal1['logfoldchanges'].abs() > log2fc)]

deg_tal1_sig_sg18 = deg_tal1_sig[deg_tal1_sig['group']=='ctrl']
deg_tal1_sig_sg24 = deg_tal1_sig[deg_tal1_sig['group']=='sAct']
deg_tal1_sig_sg26 = deg_tal1_sig[deg_tal1_sig['group']=='DR']
deg_tal1_sig_sg28 = deg_tal1_sig[deg_tal1_sig['group']=='combi']

print(deg_tal1.shape)
print(deg_tal1_fi.shape)
print(deg_tal1_sig.shape)

#### Heatmap lfc

In [ ]:
# Pivot the DataFrame to create a matrix suitable for a heatmap
heatmap_matrix = deg_tal1_sig.pivot(index='group', columns='names', values='logfoldchanges')

#figure
plt.figure(figsize=(16,4))
ax = sns.heatmap(heatmap_matrix, cmap='coolwarm', annot=False, fmt='.2f')
ax.set_title("Logfoldchanges of DEGs in TAL1")
plt.show()

####  Heatmap Gene Expression 

#### Dotplot 

In [ ]:
sc.pl.DotPlot(adata_TAL1, use_raw=True, var_names = list(set(deg_tal1_sig['names'])), groupby='sample', standard_scale="var", title='TAL1 DEGs', cmap='BuGn', figsize=(300, 5)).savefig('./figures/sig_deg_tal1_dotplot.pdf')

### IMM significant DEGs

In [ ]:
p = 0.05
log2fc = 0 

deg_imm = sc.get.rank_genes_groups_df(adata_IMM, group= None)
## If log2FoldChange > 0 and pvalue < 0.05, set as "UP"
deg_imm.loc[(deg_imm.logfoldchanges > 0) & (deg_imm.pvals_adj < 0.05), 'sign'] = "upregulated"
## If log2FoldChange < 0 and pvalue < 0.05, set as "DOWN"
deg_imm.loc[(deg_imm.logfoldchanges < 0) & (deg_imm.pvals_adj < 0.05), 'sign'] = "downregulated"

# filtered only by pval
deg_imm_fi = deg_imm[deg_imm['pvals_adj'] < p]

# significant genes -> filtered by pval and log2fc
deg_imm_sig = deg_imm[(deg_imm['pvals_adj'] < p) & (deg_imm['logfoldchanges'].abs() > log2fc)]

deg_imm_sig_sg18 = deg_imm_sig[deg_imm_sig['group']=='ctrl']
deg_imm_sig_sg24 = deg_imm_sig[deg_imm_sig['group']=='sAct']
deg_imm_sig_sg26 = deg_imm_sig[deg_imm_sig['group']=='DR']
deg_imm_sig_sg28 = deg_imm_sig[deg_imm_sig['group']=='combi']

print(deg_imm.shape)
print(deg_imm_fi.shape)
print(deg_imm_sig.shape)

#### Heatmap lfc

In [ ]:
# Pivot the DataFrame to create a matrix suitable for a heatmap
heatmap_matrix = deg_imm_sig.pivot(index='group', columns='names', values='logfoldchanges')

#figure
plt.figure(figsize=(16,6))
plt.subplots_adjust(left=0.15, bottom=0.25) 
ax = sns.heatmap(heatmap_matrix, cmap='coolwarm', annot=False, fmt='.2f')
ax.set_title("Logfoldchanges of DEGs in IMM")

plt.show()

#### Vulcanoplot

In [ ]:
# Create a column to classify genes as upregulated or downregulated
deg_imm['Regulation'] = 'Not Significant'  # Initialize as not significant
#deg_imm.loc[(deg_imm['logfoldchanges'] > 0) & (deg_imm['pvals_adj'] < 0.05), 'Regulation'] = 'Upregulated'
#deg_imm.loc[(deg_imm['logfoldchanges'] < 0) & (deg_imm['pvals_adj'] < 0.05), 'Regulation'] = 'Downregulated'
deg_imm.loc[(deg_imm['pvals_adj'] < 0.05) & (deg_imm['group'] == 'ctrl'), 'Regulation'] = 'ctrl'
deg_imm.loc[(deg_imm['pvals_adj'] < 0.05) & (deg_imm['group'] == 'sAct'), 'Regulation'] = 'sAct'
deg_imm.loc[(deg_imm['pvals_adj'] < 0.05) & (deg_imm['group'] == 'DR'), 'Regulation'] = 'DR'
deg_imm.loc[(deg_imm['pvals_adj'] < 0.05) & (deg_imm['group'] == 'combi'), 'Regulation'] = 'combi'

# Define colors for significant genes
colors = {'ctrl': 'blue','sAct': 'green','DR': 'purple','combi': 'orange', 'Not Significant': 'gray'}

plt.figure(figsize=(20, 10))
for regulation, color in colors.items():
    subset_df = deg_imm[deg_imm['Regulation'] == regulation]
    plt.scatter(subset_df['logfoldchanges'], -np.log10(subset_df['pvals_adj']), color=color, alpha=0.3, label=regulation)
   
    
# Annotate all significant genes (adjusted P-value < 0.05) with their gene names
significant_genes = deg_imm[deg_imm['pvals_adj'] < 0.05]
#for index, row in significant_genes.iterrows():
#    plt.annotate(row['names'], (row['logfoldchanges'], -np.log10(row['pvals_adj'])), color='black', fontsize=8)
    
plt.xlabel('Log2(Fold Change)')
plt.xlabel('Log Fold Change')
plt.ylabel('-log10(P-Value)')
plt.title('Volcano Plot for IMM')
plt.axvline(x=-1*log2fc, linestyle='--', color='gray')
plt.axhline(-np.log10(0.05), color='red', linestyle='--', label='Significance Threshold (Adj. p-value < 0.05)')
plt.legend()
plt.show()

#### Dotplot 

In [ ]:
sc.pl.DotPlot(adata_IMM, use_raw=True, var_names = list(set(deg_imm_sig['names'])), groupby='sample', standard_scale="var", title='IMM DEGs', cmap='BuGn', figsize=(300, 5)).savefig('./figures/sig_deg_imm_dotplot.pdf' ,format='pdf')

# Write DEG csv files

In [ ]:
# Write DR vs Age specifically
deg_podo_sig_sg26.to_csv(output_dir + 'deg_podo_sig_DRvsAge.csv', index = False)
deg_vsmc_sig_sg26.to_csv(output_dir + 'deg_vsmc_sig_DRvsAge.csv', index = False)
deg_ec1_sig_sg26.to_csv(output_dir + 'deg_ec1_sig_DRvsAge.csv', index = False)
#deg_ec1.to_csv(output_dir + 'deg_ec1.csv', index = False) 
deg_pec_sig_sg26.to_csv(output_dir + 'deg_pec_sig_DRvsAge.csv', index = False)

deg_imm_sig_sg26.to_csv(output_dir + 'deg_imm_sig_DRvsAge.csv', index = False)
deg_pt1_sig_sg26.to_csv(output_dir + 'deg_pt1_sig_DRvsAge.csv', index = False)
deg_pt2_sig_sg26.to_csv(output_dir + 'deg_pt2_sig_DRvsAge.csv', index = False)

In [ ]:
deg_podo_sig.to_csv(output_dir + 'deg_podo_sig.csv', index = False)
deg_vsmc_sig.to_csv(output_dir + 'deg_vsmc_sig.csv', index = False)
deg_ec1_sig.to_csv(output_dir + 'deg_ec1_sig.csv', index = False)
#deg_ec1.to_csv(output_dir + 'deg_ec1.csv', index = False) 
deg_pec_sig.to_csv(output_dir + 'deg_pec_sig.csv', index = False)

deg_imm_sig.to_csv(output_dir + 'deg_imm_sig.csv', index = False)
deg_pt1_sig.to_csv(output_dir + 'deg_pt1_sig.csv', index = False)
deg_pt2_sig.to_csv(output_dir + 'deg_pt2_sig.csv', index = False)

# Enrichment: Over-representation analysis (Enrichr API)

In [ ]:
from gseapy.scipalette import SciPalette
NbDr = SciPalette.create_colormap() 

## PODO

In [ ]:
# List of dataframe names
podo_names = ['deg_podo_sig_sg24', 'deg_podo_sig_sg26', 'deg_podo_sig_sg28']

### Loop Up/Downregulated KEGG

In [ ]:
# Loop over the dataframes
for df_name in podo_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
    sc.pl.DotPlot(adata_PODO, use_raw=True, var_names = df.names, groupby='sample', standard_scale="var", title=f"{df_short_name} all", cmap='BuGn', figsize=(17, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
    
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    sc.pl.DotPlot(adata_PODO, use_raw=True, var_names = degs_up.names, groupby='sample', standard_scale="var", title=f"{df_short_name} upregulated", cmap='BuGn', figsize=(15, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
    sc.pl.DotPlot(adata_PODO, use_raw=True, var_names = degs_dw.names, groupby='sample', standard_scale="var", title=f"{df_short_name} downregulated", cmap='BuGn', figsize=(15, 5)).savefig('./dotplot_dwn.pdf' ,format='pdf')
   
    
    # Check if degs_dw contains only one gene and convert it to a list
    if (degs_dw['names'].count() == 1):
        print(df_short_name)
        enr_dw = gp.enrichr(degs_dw,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
    else:
        enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
   
    # Enrichr API
    enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)

    
    enr_full = gp.enrichr(df.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)

   # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG upregulated", cmap=plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG downregulated", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        #gp.dotplot(enr_full.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG all", cmap=plt.cm.winter_r, size=5)
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} KEGG all", cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()


    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-KEGG",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-KEGG",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()

### Loop Up/Downregulated GO BP

In [ ]:
# Loop over the dataframes
for df_name in podo_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
  
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    # Enrichr API
    enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)

    enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
    
    enr_full = gp.enrichr(df.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        organism='Mouse',
                        outdir=None)
    
    # trim (go:...)
    enr_up.res2d.Term = enr_up.res2d.Term.str.split(" \(GO").str[0]
    enr_dw.res2d.Term = enr_dw.res2d.Term.str.split(" \(GO").str[0]
    
    # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3,5), title=f"{df_short_name} GO BP upregulated", cmap = plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3,5), title=f"{df_short_name} GO BP downregulated", cmap = plt.cm.winter_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} GO BP all")
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()

    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-GO_BP",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-GO_BP",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()

### SG24

In [ ]:
# subset up or down regulated genes
deg_podo_sig_sg24['sign'] = np.where(deg_podo_sig_sg24['logfoldchanges'] > 0, 'upregulated', 'downregulated')

degs_up = deg_podo_sig_sg24[deg_podo_sig_sg24['logfoldchanges'] > 0]
degs_dw = deg_podo_sig_sg24[deg_podo_sig_sg24['logfoldchanges'] < 0]

# Enricr API
enr_up = gp.enrichr(degs_up.names,
                    gene_sets=['GO_Biological_Process_2021','KEGG_2019_Mouse'],
                    outdir=None)

enr_dw = gp.enrichr(degs_dw.names,
                    gene_sets='GO_Biological_Process_2021',
                    outdir=None)

# trim (go:...)
enr_up.res2d.Term = enr_up.res2d.Term.str.split(" \(GO").str[0]
enr_dw.res2d.Term = enr_dw.res2d.Term.str.split(" \(GO").str[0]

# dotplots
gp.dotplot(enr_up.res2d, figsize=(3,5), title="PODO SG24 Up", cmap = plt.cm.autumn_r)
plt.show()
gp.dotplot(enr_dw.res2d,   figsize=(3,5), title="PODO SG24 Down", cmap = plt.cm.winter_r, cutoff = 0.2)
plt.show()


# concat results
enr_up.res2d['UP_DW'] = "UP"
enr_dw.res2d['UP_DW'] = "DOWN"
enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

# display multi-datasets
ax = gp.dotplot(enr_res,figsize=(10,5),
                x='UP_DW',
                x_order = ["UP", "DOWN"],
                title="PODO SG24 GO_BP",
                cmap = NbDr,
                size=3,
                show_ring=True)
ax.set_xlabel("")
plt.show()

ax = gp.barplot(enr_res, figsize=(3,5),
                group ='UP_DW',
                title ="PODO SG24 GO_BP",
                color = ['b','r'])

In [ ]:
glist_sg24 = deg_podo_sig_sg24['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

geneset_kegg = enrichr_results.results['Genes'] # this column contains the genes for each significant pathway
geneset_gobp = enrichr_results_gobp.results['Genes'] 

gs_kegg_podoSG24_list = [genes.split(';') for genes in geneset_kegg]
gs_gobp_podoSG24_list = [genes.split(';') for genes in geneset_gobp]

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_podo_sg24.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_podo_sg24.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG24 KEGG Enrichment analysis PODO')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG24 KEGG Enrichment analysis PODO', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG24 GO BP Enrichment analysis PODO')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG24 GO BP Enrichment analysis PODO', cmap='RdBu_r')

### SG26

In [ ]:
# subset up or down regulated genes
deg_podo_sig_sg26['sign'] = np.where(deg_podo_sig_sg26['logfoldchanges'] > 0, 'upregulated', 'downregulated')

degs_up = deg_podo_sig_sg26[deg_podo_sig_sg26['logfoldchanges'] > 0]
degs_dw = deg_podo_sig_sg26[deg_podo_sig_sg26['logfoldchanges'] < 0]

# Enricr API
enr_up = gp.enrichr(degs_up.names,
                    gene_sets=['GO_Biological_Process_2021','KEGG_2019_Mouse'],
                    outdir=None)

enr_dw = gp.enrichr(degs_dw.names,
                    gene_sets='GO_Biological_Process_2021',
                    outdir=None)

# trim (go:...)
enr_up.res2d.Term = enr_up.res2d.Term.str.split(" \(GO").str[0]
enr_dw.res2d.Term = enr_dw.res2d.Term.str.split(" \(GO").str[0]

# dotplots
gp.dotplot(enr_up.res2d, figsize=(3,5), title="PODO SG26 Up", cmap = plt.cm.autumn_r)
plt.show()
gp.dotplot(enr_dw.res2d,   figsize=(3,5), title="PODO SG26 Down", cmap = plt.cm.winter_r, cutoff = 0.2)
plt.show()


# concat results
enr_up.res2d['UP_DW'] = "UP"
enr_dw.res2d['UP_DW'] = "DOWN"
enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

# display multi-datasets
ax = gp.dotplot(enr_res,figsize=(10,5),
                x='UP_DW',
                x_order = ["UP", "DOWN"],
                title="PODO SG26 GO_BP",
                cmap = NbDr,
                size=3,
                show_ring=True)
ax.set_xlabel("")
plt.show()

ax = gp.barplot(enr_res, figsize=(3,5),
                group ='UP_DW',
                title ="PODO SG26 GO_BP",
                color = ['b','r'])

In [ ]:
len(deg_podo_sig_sg26) 

glist_sg26 = deg_podo_sig_sg26['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg26,organism='Mouse',gene_sets='KEGG_2019_Mouse', cutoff=0.05)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

geneset_kegg = enrichr_results.results['Genes'] # this column contains the genes for each significant pathway
geneset_gobp = enrichr_results_gobp.results['Genes'] 

gs_kegg_podoSG26_list = [genes.split(';') for genes in geneset_kegg]
gs_gobp_podoSG26_list = [genes.split(';') for genes in geneset_gobp]

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_podo_sg26.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_podo_sg26.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG26 KEGG Enrichment analysis PODO')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG26 KEGG Enrichment analysis PODO', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG26 GO BP Enrichment analysis PODO')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG26 GO BP Enrichment analysis PODO', cmap='RdBu_r')

### SG28

In [ ]:
# subset up or down regulated genes
deg_podo_sig_sg28['sign'] = np.where(deg_podo_sig_sg28['logfoldchanges'] > 0, 'upregulated', 'downregulated')

degs_up = deg_podo_sig_sg28[deg_podo_sig_sg28['logfoldchanges'] > 0]
degs_dw = deg_podo_sig_sg28[deg_podo_sig_sg28['logfoldchanges'] < 0]

# Enricr API
enr_up = gp.enrichr(degs_up.names,
                    gene_sets=['GO_Biological_Process_2021','KEGG_2019_Mouse'],
                    outdir=None)

enr_dw = gp.enrichr(degs_dw.names,
                    gene_sets='GO_Biological_Process_2021',
                    outdir=None)

# trim (go:...)
enr_up.res2d.Term = enr_up.res2d.Term.str.split(" \(GO").str[0]
enr_dw.res2d.Term = enr_dw.res2d.Term.str.split(" \(GO").str[0]

# dotplots
gp.dotplot(enr_up.res2d, figsize=(3,5), title="PODO SG28 Up", cmap = plt.cm.autumn_r)
plt.show()
gp.dotplot(enr_dw.res2d,   figsize=(3,5), title="PODO SG28 Down", cmap = plt.cm.winter_r, cutoff = 0.2)
plt.show()


# concat results
enr_up.res2d['UP_DW'] = "UP"
enr_dw.res2d['UP_DW'] = "DOWN"
enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

# display multi-datasets
ax = gp.dotplot(enr_res,figsize=(10,5),
                x='UP_DW',
                x_order = ["UP", "DOWN"],
                title="PODO SG28 GO_BP",
                cmap = NbDr,
                size=3,
                show_ring=True)
ax.set_xlabel("")
plt.show()

ax = gp.barplot(enr_res, figsize=(3,5),
                group ='UP_DW',
                title ="PODO SG28 GO_BP",
                color = ['b','r'])

In [ ]:
len(deg_podo_sig_sg28) 

glist_sg28 = deg_podo_sig_sg28['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

geneset_kegg = enrichr_results.results['Genes'] # this column contains the genes for each significant pathway
geneset_gobp = enrichr_results_gobp.results['Genes'] 

gs_kegg_podoSG28_list = [genes.split(';') for genes in geneset_kegg]
gs_gobp_podoSG28_list = [genes.split(';') for genes in geneset_gobp]

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_podo_sg28.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_podo_sg28.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG28 KEGG Enrichment analysis PODO')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG28 KEGG Enrichment analysis PODO', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG28 GO BP Enrichment analysis PODO')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG28 GO BP Enrichment analysis PODO', cmap='RdBu_r')

## vSMC

In [ ]:
# List of dataframe names
vsmc_names = ['deg_vsmc_sig_sg24', 'deg_vsmc_sig_sg26', 'deg_vsmc_sig_sg28']

### Loop Up/Downregulated KEGG

In [ ]:
# Loop over the dataframes
for df_name in vsmc_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
    sc.pl.DotPlot(adata_vSMC, use_raw=True, var_names = df.names, groupby='sample', standard_scale="var", title=f"{df_short_name} all", cmap='BuGn', figsize=(17, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
    
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    sc.pl.DotPlot(adata_vSMC, use_raw=True, var_names = degs_up.names, groupby='sample', standard_scale="var", title=f"{df_short_name} upregulated", cmap='BuGn', figsize=(15, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
    sc.pl.DotPlot(adata_vSMC, use_raw=True, var_names = degs_dw.names, groupby='sample', standard_scale="var", title=f"{df_short_name} downregulated", cmap='BuGn', figsize=(15, 5)).savefig('./dotplot_dwn.pdf' ,format='pdf')
   
    
    # Check if degs_dw contains only one gene and convert it to a list
    if (degs_dw['names'].count() == 1):
        print(df_short_name)
        enr_dw = gp.enrichr(degs_dw,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
    else:
        enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
   
    # Enrichr API
    enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)

    
    enr_full = gp.enrichr(df.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)

   # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG upregulated", cmap=plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG downregulated", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        #gp.dotplot(enr_full.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG all", cmap=plt.cm.winter_r, size=5)
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} KEGG all", cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()


    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-KEGG",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-KEGG",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()

### Loop Up/Downregulated GO BP

In [ ]:
# Loop over the dataframes
for df_name in vsmc_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
  
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    # Enrichr API
    enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)

    enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
    
    enr_full = gp.enrichr(df.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        organism='Mouse',
                        outdir=None)
    
    # trim (go:...)
    enr_up.res2d.Term = enr_up.res2d.Term.str.split(" \(GO").str[0]
    enr_dw.res2d.Term = enr_dw.res2d.Term.str.split(" \(GO").str[0]
    
    # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3,5), title=f"{df_short_name} GO BP upregulated", cmap = plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3,5), title=f"{df_short_name} GO BP downregulated", cmap = plt.cm.winter_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} GO BP all")
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()

    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-GO_BP",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-GO_BP",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()

### SG24

In [ ]:
glist_sg24 = deg_vsmc_sig_sg24['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

geneset_kegg = enrichr_results.results['Genes'] # this column contains the genes for each significant pathway
geneset_gobp = enrichr_results_gobp.results['Genes'] 

gs_kegg_vsmcSG24_list = [genes.split(';') for genes in geneset_kegg]
gs_gobp_vsmcSG24_list = [genes.split(';') for genes in geneset_gobp]

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_vsmc_sg24.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_vsmc_sg24.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG24 KEGG Enrichment analysis vSMC')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG24 KEGG Enrichment analysis vSMC', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG24 GO BP Enrichment analysis vSMC')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG24 GO BP Enrichment analysis vSMC', cmap='RdBu_r')

### SG26

In [ ]:
len(deg_vsmc_sig_sg26) 

glist_sg26 = deg_vsmc_sig_sg26['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg26,organism='Mouse',gene_sets='KEGG_2019_Mouse',cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

geneset_kegg = enrichr_results.results['Genes'] # this column contains the genes for each significant pathway
geneset_gobp = enrichr_results_gobp.results['Genes'] 

gs_kegg_vsmcSG26_list = [genes.split(';') for genes in geneset_kegg]
gs_gobp_vsmcSG26_list = [genes.split(';') for genes in geneset_gobp]

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_vsmc_sg26.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_vsmc_sg26.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG26 KEGG Enrichment analysis vSMC')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG26 KEGG Enrichment analysis vSMC', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG26 GO BP Enrichment analysis vSMC')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG26 GO BP Enrichment analysis vSMC', cmap='RdBu_r')

### SG28

In [ ]:
len(deg_vsmc_sig_sg28) 

glist_sg28 = deg_vsmc_sig_sg28['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

geneset_kegg = enrichr_results.results['Genes'] # this column contains the genes for each significant pathway
geneset_gobp = enrichr_results_gobp.results['Genes'] 

gs_kegg_vsmcSG28_list = [genes.split(';') for genes in geneset_kegg]
gs_gobp_vsmcSG28_list = [genes.split(';') for genes in geneset_gobp]

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_vsmc_sg28.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_vsmc_sg28.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG28 KEGG Enrichment analysis vSMC')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG28 KEGG Enrichment analysis vSMC', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG28 GO BP Enrichment analysis vSMC')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG28 GO BP Enrichment analysis vSMC', cmap='RdBu_r')

## PEC

In [ ]:
# List of dataframe names
pec_names = ['deg_pec_sig_sg24', 'deg_pec_sig_sg26', 'deg_pec_sig_sg28']

### Loop Up/Downregulated KEGG

In [ ]:
# Loop over the dataframes
for df_name in pec_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
    sc.pl.DotPlot(adata_PEC, use_raw=True, var_names = df.names, groupby='sample', standard_scale="var", title=f"{df_short_name} all", cmap='BuGn', figsize=(17, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
    
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    sc.pl.DotPlot(adata_PEC, use_raw=True, var_names = degs_up.names, groupby='sample', standard_scale="var", title=f"{df_short_name} upregulated", cmap='BuGn', figsize=(15, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
    sc.pl.DotPlot(adata_PEC, use_raw=True, var_names = degs_dw.names, groupby='sample', standard_scale="var", title=f"{df_short_name} downregulated", cmap='BuGn', figsize=(15, 5)).savefig('./dotplot_dwn.pdf' ,format='pdf')
   
    
    # Check if degs_dw contains only one gene and convert it to a list
    if (degs_dw['names'].count() == 1):
        print(df_short_name)
        enr_dw = gp.enrichr(degs_dw,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
    else:
        enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
   
    # Enrichr API
    enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)

    
    enr_full = gp.enrichr(df.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)

   # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG upregulated", cmap=plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG downregulated", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        #gp.dotplot(enr_full.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG all", cmap=plt.cm.winter_r, size=5)
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} KEGG all", cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()


    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-KEGG",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-KEGG",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()

### Loop Up/Downregulated GO BP

In [ ]:
# Loop over the dataframes
for df_name in pec_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
  
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    # Enrichr API
    enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)

    enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
    
    enr_full = gp.enrichr(df.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        organism='Mouse',
                        outdir=None)
    
    # trim (go:...)
    enr_up.res2d.Term = enr_up.res2d.Term.str.split(" \(GO").str[0]
    enr_dw.res2d.Term = enr_dw.res2d.Term.str.split(" \(GO").str[0]
    
    # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3,5), title=f"{df_short_name} GO BP upregulated", cmap = plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3,5), title=f"{df_short_name} GO BP downregulated", cmap = plt.cm.winter_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} GO BP all", cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()

    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-GO_BP",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-GO_BP",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()

In [ ]:
glist_sg18 = deg_pec_sig_sg18['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg18,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg18,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_pec_sg18.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_pec_sg18.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG18 KEGG Enrichment analysis PEC')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG18 KEGG Enrichment analysis PEC', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG18 GO BP Enrichment analysis PEC')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG18 GO BP Enrichment analysis PEC', cmap='RdBu_r')

### SG24

In [ ]:
glist_sg24 = deg_pec_sig_sg24['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

geneset_kegg = enrichr_results.results['Genes'] # this column contains the genes for each significant pathway
geneset_gobp = enrichr_results_gobp.results['Genes'] 

gs_kegg_pecSG24_list = [genes.split(';') for genes in geneset_kegg]
gs_gobp_pecSG24_list = [genes.split(';') for genes in geneset_gobp]

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_pec_sg24.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_pec_sg24.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG24 KEGG Enrichment analysis PEC', cutoff = 0.5)
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG24 KEGG Enrichment analysis PEC', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG24 GO BP Enrichment analysis PEC')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG24 GO BP Enrichment analysis PEC', cmap='RdBu_r')

### SG26

In [ ]:
glist_sg26 = deg_pec_sig_sg26['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg26,organism='Mouse',gene_sets='KEGG_2019_Mouse',cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

geneset_kegg = enrichr_results.results['Genes'] # this column contains the genes for each significant pathway
geneset_gobp = enrichr_results_gobp.results['Genes'] 

gs_kegg_pecSG26_list = [genes.split(';') for genes in geneset_kegg]
gs_gobp_pecSG26_list = [genes.split(';') for genes in geneset_gobp]

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_pec_sg26.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_pec_sg26.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG26 KEGG Enrichment analysis PEC')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG26 KEGG Enrichment analysis PEC', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG26 GO BP Enrichment analysis PEC', cutoff = 0.5)
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG26 GO BP Enrichment analysis PEC', cmap='RdBu_r')

### SG28

In [ ]:
glist_sg28 = deg_pec_sig_sg28['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

geneset_kegg = enrichr_results.results['Genes'] # this column contains the genes for each significant pathway
geneset_gobp = enrichr_results_gobp.results['Genes'] 

gs_kegg_pecSG28_list = [genes.split(';') for genes in geneset_kegg]
gs_gobp_pecSG28_list = [genes.split(';') for genes in geneset_gobp]

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_pec_sg28.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_pec_sg28.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG28 KEGG Enrichment analysis PEC')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG28 KEGG Enrichment analysis PEC', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG28 GO BP Enrichment analysis PEC')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG28 GO BP Enrichment analysis PEC', cmap='RdBu_r')

## EC1

In [ ]:
# List of dataframe names
ec1_names = ['deg_ec1_sig_sg24', 'deg_ec1_sig_sg26', 'deg_ec1_sig_sg28']

### Loop Up/Downregulated KEGG

In [ ]:
# Loop over the dataframes
for df_name in ec1_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
    sc.pl.DotPlot(adata_EC1, use_raw=True, var_names = df.names, groupby='sample', standard_scale="var", title=f"{df_short_name} all", cmap='BuGn', figsize=(17, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
    
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    sc.pl.DotPlot(adata_EC1, use_raw=True, var_names = degs_up.names, groupby='sample', standard_scale="var", title=f"{df_short_name} upregulated", cmap='BuGn', figsize=(15, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
    sc.pl.DotPlot(adata_EC1, use_raw=True, var_names = degs_dw.names, groupby='sample', standard_scale="var", title=f"{df_short_name} downregulated", cmap='BuGn', figsize=(15, 5)).savefig('./dotplot_dwn.pdf' ,format='pdf')
   
    # Enrichr API
    # Check if degs_dw contains only one gene and convert it to a list
    if (degs_dw['names'].count() == 1):
        print(df_short_name)
        enr_dw = gp.enrichr(degs_dw,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
    else:
        enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
        
     # Check if degs_dw contains only one gene and convert it to a list
    if (degs_up['names'].count() == 1):
        print(df_short_name)
        enr_up = gp.enrichr(degs_up,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
    else:
        enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)   
   
    enr_full = gp.enrichr(df.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)

   # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG upregulated", cmap=plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG downregulated", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        #gp.dotplot(enr_full.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG all", cmap=plt.cm.winter_r, size=5)
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} KEGG all", cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()


    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-KEGG",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-KEGG",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()

### Loop Up/Downregulated GO BP

In [ ]:
# Loop over the dataframes
for df_name in ec1_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
  
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    # Enrichr API
     # Enrichr API
    # Check if degs_dw contains only one gene and convert it to a list
    if (degs_dw['names'].count() == 1):
        print(df_short_name)
        enr_dw = gp.enrichr(degs_dw,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
    else:
        enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
        
     # Check if degs_dw contains only one gene and convert it to a list
    if (degs_up['names'].count() == 1):
        print(df_short_name)
        enr_up = gp.enrichr(degs_up,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
    else:
        enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)   
   
    enr_full = gp.enrichr(df.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        organism='Mouse',
                        outdir=None)
    
    
    # trim (go:...)
    enr_up.res2d.Term = enr_up.res2d.Term.str.split(" \(GO").str[0]
    enr_dw.res2d.Term = enr_dw.res2d.Term.str.split(" \(GO").str[0]
    
    # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3,5), title=f"{df_short_name} GO BP upregulated", cmap = plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3,5), title=f"{df_short_name} GO BP downregulated", cmap = plt.cm.winter_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} GO BP all", cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()

    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-GO_BP",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-GO_BP",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()

In [ ]:
glist_sg18 = deg_ec1_sig_sg18['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg18,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse')

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg18,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_ec1_sg18.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_ec1_sg18.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG18 KEGG Enrichment analysis EC1', cutoff = 0.5)
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG18 KEGG Enrichment analysis EC1', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG18 GO BP Enrichment analysis EC1')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG18 GO BP Enrichment analysis EC1', cmap='RdBu_r')

### SG24

In [ ]:
glist_sg24 = deg_ec1_sig_sg24['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse')

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

geneset_kegg = enrichr_results.results['Genes'] # this column contains the genes for each significant pathway
geneset_gobp = enrichr_results_gobp.results['Genes'] 

gs_kegg_ec1SG24_list = [genes.split(';') for genes in geneset_kegg]
gs_gobp_ec1SG24_list = [genes.split(';') for genes in geneset_gobp]

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_ec1_sg24.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_ec1_sg24.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG24 KEGG Enrichment analysis EC1', cutoff = 0.05)
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG24 KEGG Enrichment analysis EC1', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG24 GO BP Enrichment analysis EC1')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG24 GO BP Enrichment analysis EC1', cmap='RdBu_r')

### SG26

In [ ]:
len(deg_ec1_sig_sg26) 

glist_sg26 = deg_ec1_sig_sg26['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg26,organism='Mouse',gene_sets='KEGG_2019_Mouse')

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

geneset_kegg = enrichr_results.results['Genes'] # this column contains the genes for each significant pathway
geneset_gobp = enrichr_results_gobp.results['Genes'] 

gs_kegg_ec1SG26_list = [genes.split(';') for genes in geneset_kegg]
gs_gobp_ec1SG26_list = [genes.split(';') for genes in geneset_gobp]

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_ec1_sg26.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_ec1_sg26.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG26 KEGG Enrichment analysis EC1')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG26 KEGG Enrichment analysis EC1', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG26 GO BP Enrichment analysis EC1')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG26 GO BP Enrichment analysis EC1', cmap='RdBu_r')

### SG28

In [ ]:
len(deg_ec1_sig_sg28) 

glist_sg28 = deg_ec1_sig_sg28['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse')

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

geneset_kegg = enrichr_results.results['Genes'] # this column contains the genes for each significant pathway
geneset_gobp = enrichr_results_gobp.results['Genes'] 

gs_kegg_ec1SG28_list = [genes.split(';') for genes in geneset_kegg]
gs_gobp_ec1SG28_list = [genes.split(';') for genes in geneset_gobp]

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_ec1_sg28.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_ec1_sg28.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG28 KEGG Enrichment analysis EC1', cutoff = 0.5)
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG28 KEGG Enrichment analysis EC1', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG28 GO BP Enrichment analysis EC1', cutoff = 0.5)
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG28 GO BP Enrichment analysis EC1', cmap='RdBu_r')

## PT1

In [ ]:
# List of dataframe names
pt1_names = ['deg_pt1_sig_sg24', 'deg_pt1_sig_sg26', 'deg_pt1_sig_sg28']

### Loop Up/Downregulated KEGG

In [ ]:
# Loop over the dataframes
for df_name in pt1_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
    sc.pl.DotPlot(adata_PT1, use_raw=True, var_names = df.names, groupby='sample', standard_scale="var", title=f"{df_short_name} all", cmap='BuGn', figsize=(17, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
    
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    sc.pl.DotPlot(adata_PT1, use_raw=True, var_names = degs_up.names, groupby='sample', standard_scale="var", title=f"{df_short_name} upregulated", cmap='BuGn', figsize=(15, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
    sc.pl.DotPlot(adata_PT1, use_raw=True, var_names = degs_dw.names, groupby='sample', standard_scale="var", title=f"{df_short_name} downregulated", cmap='BuGn', figsize=(15, 5)).savefig('./dotplot_dwn.pdf' ,format='pdf')
   
    # Enrichr API
    # Check if degs_dw contains only one gene and convert it to a list
    if (degs_dw['names'].count() == 1):
        print(df_short_name)
        enr_dw = gp.enrichr(degs_dw,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
    else:
        enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
        
     # Check if degs_dw contains only one gene and convert it to a list
    if (degs_up['names'].count() == 1):
        print(df_short_name)
        enr_up = gp.enrichr(degs_up,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
    else:
        enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)   
   
    enr_full = gp.enrichr(df.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)

   # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG upregulated", cmap=plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG downregulated", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        #gp.dotplot(enr_full.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG all", cmap=plt.cm.winter_r, size=5)
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} KEGG all", cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()


    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-KEGG",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-KEGG",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()

### Loop Up/Downregulated GO BP

In [ ]:
# Loop over the dataframes
for df_name in pt1_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
  
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    # Enrichr API
     # Enrichr API
    # Check if degs_dw contains only one gene and convert it to a list
    if (degs_dw['names'].count() == 1):
        print(df_short_name)
        enr_dw = gp.enrichr(degs_dw,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
    else:
        enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
        
     # Check if degs_dw contains only one gene and convert it to a list
    if (degs_up['names'].count() == 1):
        print(df_short_name)
        enr_up = gp.enrichr(degs_up,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
    else:
        enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)   
   
    enr_full = gp.enrichr(df.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        organism='Mouse',
                        outdir=None)
    
    
    # trim (go:...)
    enr_up.res2d.Term = enr_up.res2d.Term.str.split(" \(GO").str[0]
    enr_dw.res2d.Term = enr_dw.res2d.Term.str.split(" \(GO").str[0]
    
    # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3,5), title=f"{df_short_name} GO BP upregulated", cmap = plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3,5), title=f"{df_short_name} GO BP downregulated", cmap = plt.cm.winter_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} GO BP all", cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()

    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-GO_BP",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-GO_BP",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()

In [ ]:
glist_sg18 = deg_pt1_sig_sg18['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg18,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.05)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg18,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_pt1_sg18.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_pt1_sg18.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG18 KEGG Enrichment analysis PT1')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG18 KEGG Enrichment analysis PT1', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG18 GO BP Enrichment analysis PT1')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG18 GO BP Enrichment analysis PT1', cmap='RdBu_r')

### SG24

In [ ]:
glist_sg24 = deg_pt1_sig_sg24['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

geneset_kegg = enrichr_results.results['Genes'] # this column contains the genes for each significant pathway
geneset_gobp = enrichr_results_gobp.results['Genes'] 

gs_kegg_pt1SG24_list = [genes.split(';') for genes in geneset_kegg]
gs_gobp_pt1SG24_list = [genes.split(';') for genes in geneset_gobp]


enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_pt1_sg24.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_pt1_sg24.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG24 KEGG Enrichment analysis PT1')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG24 KEGG Enrichment analysis PT1', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG24 GO BP Enrichment analysis PT1')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG24 GO BP Enrichment analysis PT1', cmap='RdBu_r')

### SG26

In [ ]:
len(deg_pt1_sig_sg26) 

glist_sg26 = deg_pt1_sig_sg26['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse')


enrichr_results_gobp = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

geneset_kegg = enrichr_results.results['Genes'] # this column contains the genes for each significant pathway
geneset_gobp = enrichr_results_gobp.results['Genes'] 

gs_kegg_pt1SG26_list = [genes.split(';') for genes in geneset_kegg]
gs_gobp_pt1SG26_list = [genes.split(';') for genes in geneset_gobp]

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_pt1_sg26.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_pt1_sg26.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG26 KEGG Enrichment analysis PT1')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG26 KEGG Enrichment analysis PT1', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG26 GO BP Enrichment analysis PT1', cutoff = 0.05)
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG26 GO BP Enrichment analysis PT1', cmap='RdBu_r')

### SG28

In [ ]:
len(deg_pt1_sig_sg28) 

glist_sg28 = deg_pt1_sig_sg28['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg28,organism='Mouse',gene_sets='KEGG_2019_Mouse')


enrichr_results_gobp = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

geneset_kegg = enrichr_results.results['Genes'] # this column contains the genes for each significant pathway
geneset_gobp = enrichr_results_gobp.results['Genes'] 

gs_kegg_pt1SG28_list = [genes.split(';') for genes in geneset_kegg]
gs_gobp_pt1SG28_list = [genes.split(';') for genes in geneset_gobp]

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_pt1_sg28.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_pt1_sg28.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG28 KEGG Enrichment analysis PT1')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG28 KEGG Enrichment analysis PT1', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG28 GO BP Enrichment analysis PT1')
dotplot(enrichr_results_gobp.results, cutoff=0.8, top_term=20, title='SG28 GO BP Enrichment analysis PT1', cmap='RdBu_r')

## PT2

In [ ]:
# List of dataframe names
pt2_names = ['deg_pt2_sig_sg24', 'deg_pt2_sig_sg26', 'deg_pt2_sig_sg28']

### Loop Up/Downregulated KEGG

In [ ]:
# Loop over the dataframes
for df_name in pt2_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
    sc.pl.DotPlot(adata_PT2, use_raw=True, var_names = df.names, groupby='sample', standard_scale="var", title=f"{df_short_name} all", cmap='BuGn', figsize=(17, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
    
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    sc.pl.DotPlot(adata_PT2, use_raw=True, var_names = degs_up.names, groupby='sample', standard_scale="var", title=f"{df_short_name} upregulated", cmap='BuGn', figsize=(15, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
    sc.pl.DotPlot(adata_PT2, use_raw=True, var_names = degs_dw.names, groupby='sample', standard_scale="var", title=f"{df_short_name} downregulated", cmap='BuGn', figsize=(15, 5)).savefig('./dotplot_dwn.pdf' ,format='pdf')
   
    # Enrichr API
    # Check if degs_dw contains only one gene and convert it to a list
    if (degs_dw['names'].count() == 1):
        print(df_short_name)
        enr_dw = gp.enrichr(degs_dw,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
    else:
        enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
        
     # Check if degs_dw contains only one gene and convert it to a list
    if (degs_up['names'].count() == 1):
        print(df_short_name)
        enr_up = gp.enrichr(degs_up,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
    else:
        enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)   
   
    enr_full = gp.enrichr(df.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)

   # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG upregulated", cmap=plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG downregulated", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        #gp.dotplot(enr_full.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG all", cmap=plt.cm.winter_r, size=5)
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} KEGG all", cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()


    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-KEGG",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-KEGG",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()

### Loop Up/Downregulated GO BP

In [ ]:
# Loop over the dataframes
for df_name in pt2_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
  
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    # Enrichr API
     # Enrichr API
    # Check if degs_dw contains only one gene and convert it to a list
    if (degs_dw['names'].count() == 1):
        print(df_short_name)
        enr_dw = gp.enrichr(degs_dw,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
    else:
        enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
        
     # Check if degs_dw contains only one gene and convert it to a list
    if (degs_up['names'].count() == 1):
        print(df_short_name)
        enr_up = gp.enrichr(degs_up,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
    else:
        enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)   
   
    enr_full = gp.enrichr(df.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        organism='Mouse',
                        outdir=None)
    
    
    # trim (go:...)
    enr_up.res2d.Term = enr_up.res2d.Term.str.split(" \(GO").str[0]
    enr_dw.res2d.Term = enr_dw.res2d.Term.str.split(" \(GO").str[0]
    
    # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3,5), title=f"{df_short_name} GO BP upregulated", cmap = plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3,5), title=f"{df_short_name} GO BP downregulated", cmap = plt.cm.winter_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} GO BP all", cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()

    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-GO_BP",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-GO_BP",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()

In [ ]:
glist_sg18 = deg_pt2_sig_sg18['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg18,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.05)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg18,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_pt2_sg18.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_pt2_sg18.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG18 KEGG Enrichment analysis PT2')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG18 KEGG Enrichment analysis PT2', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG18 GO BP Enrichment analysis PT2')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG18 GO BP Enrichment analysis PT2', cmap='RdBu_r')

### SG24

In [ ]:
glist_sg24 = deg_pt2_sig_sg24['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

geneset_kegg = enrichr_results.results['Genes'] # this column contains the genes for each significant pathway
geneset_gobp = enrichr_results_gobp.results['Genes'] 

gs_kegg_pt2SG24_list = [genes.split(';') for genes in geneset_kegg]
gs_gobp_pt2SG24_list = [genes.split(';') for genes in geneset_gobp]

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_pt2_sg24.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_pt2_sg24.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG24 KEGG Enrichment analysis PT2')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG24 KEGG Enrichment analysis PT2', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG24 GO BP Enrichment analysis PT2')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG24 GO BP Enrichment analysis PT2', cmap='RdBu_r')

### SG26

In [ ]:
len(deg_pt2_sig_sg26) 

glist_sg26 = deg_pt2_sig_sg26['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse')


enrichr_results_gobp = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

geneset_kegg = enrichr_results.results['Genes'] # this column contains the genes for each significant pathway
geneset_gobp = enrichr_results_gobp.results['Genes'] 

gs_kegg_pt2SG26_list = [genes.split(';') for genes in geneset_kegg]
gs_gobp_pt2SG26_list = [genes.split(';') for genes in geneset_gobp]

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_pt2_sg26.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_pt2_sg26.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG26 KEGG Enrichment analysis PT2')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG26 KEGG Enrichment analysis PT2', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG26 GO BP Enrichment analysis PT2', cutoff = 0.05)
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG26 GO BP Enrichment analysis PT2', cmap='RdBu_r')

### SG28

In [ ]:
len(deg_pt2_sig_sg28) 

glist_sg28 = deg_pt2_sig_sg28['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg28,organism='Mouse',gene_sets='KEGG_2019_Mouse')


enrichr_results_gobp = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

geneset_kegg = enrichr_results.results['Genes'] # this column contains the genes for each significant pathway
geneset_gobp = enrichr_results_gobp.results['Genes'] 

gs_kegg_pt2SG28_list = [genes.split(';') for genes in geneset_kegg]
gs_gobp_pt2SG28_list = [genes.split(';') for genes in geneset_gobp]

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_pt2_sg28.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_pt2_sg28.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG28 KEGG Enrichment analysis PT2')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG28 KEGG Enrichment analysis PT2', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG28 GO BP Enrichment analysis PT2')
dotplot(enrichr_results_gobp.results, cutoff=0.8, top_term=20, title='SG28 GO BP Enrichment analysis PT2', cmap='RdBu_r')

## EC2

In [ ]:
glist_sg18 = deg_ec2_sig_sg18['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg18,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg18,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_ec2_sg18.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_ec2_sg18.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG18 KEGG Enrichment analysis EC2')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG18 KEGG Enrichment analysis EC2', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG18 GO BP Enrichment analysis EC2')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG18 GO BP Enrichment analysis EC2', cmap='RdBu_r')

### SG24

In [ ]:
glist_sg24 = deg_ec2_sig_sg24['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

geneset_kegg = enrichr_results.results['Genes'] # this column contains the genes for each significant pathway
geneset_gobp = enrichr_results_gobp.results['Genes'] 

gs_kegg_ec2SG24_list = [genes.split(';') for genes in geneset_kegg]
gs_gobp_ec2SG24_list = [genes.split(';') for genes in geneset_gobp]

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_ec2_sg24.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_ec2_sg24.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG24 KEGG Enrichment analysis EC2')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG24 KEGG Enrichment analysis EC2', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG24 GO BP Enrichment analysis EC2')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG24 GO BP Enrichment analysis EC2', cmap='RdBu_r')

### SG26

In [ ]:
len(deg_ec2_sig_sg26) 

glist_sg26 = deg_ec2_sig_sg26['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

geneset_kegg = enrichr_results.results['Genes'] # this column contains the genes for each significant pathway
geneset_gobp = enrichr_results_gobp.results['Genes'] 

gs_kegg_ec2SG26_list = [genes.split(';') for genes in geneset_kegg]
gs_gobp_ec2SG26_list = [genes.split(';') for genes in geneset_gobp]


enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_ec2_sg26.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_ec2_sg26.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG26 KEGG Enrichment analysis EC2')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG26 KEGG Enrichment analysis EC2', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG26 GO BP Enrichment analysis EC2')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG26 GO BP Enrichment analysis EC2', cmap='RdBu_r')

### SG28

In [ ]:
len(deg_ec2_sig_sg28) 

glist_sg28 = deg_ec2_sig_sg28['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

geneset_kegg = enrichr_results.results['Genes'] # this column contains the genes for each significant pathway
geneset_gobp = enrichr_results_gobp.results['Genes'] 

gs_kegg_ec2SG28_list = [genes.split(';') for genes in geneset_kegg]
gs_gobp_ec2SG28_list = [genes.split(';') for genes in geneset_gobp]

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_ec2_sg28.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_ec2_sg28.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG28 KEGG Enrichment analysis EC2')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG28 KEGG Enrichment analysis EC2', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG28 GO BP Enrichment analysis EC2')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG28 GO BP Enrichment analysis EC2', cmap='RdBu_r')

In [ ]:
glist_sg18 = deg_tal1_sig_sg18['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
#enrichr_results = gp.enrichr(gene_list= glist_sg18, organism='Mouse', gene_sets='KEGG_2019_Mouse'# background='mmusculus_gene_ensembl')

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg18,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

#enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_tal1_sg18.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_tal1_sg18.csv')

# Visualize the results
#gsplot.barplot(enrichr_results.res2d, title='SG18 Enrichment Analysis Results')
#dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG18 Enrichment analysis TAL1', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG18 GO BP Enrichment analysis TAL1')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG18 GO BP Enrichment analysis TAL1', cmap='RdBu_r')

### SG24

In [ ]:
glist_sg24 = deg_tal1_sig_sg24['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_tal1_sg24.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_tal1_sg24.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG24 KEGG Enrichment analysis TAL1')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG24 KEGG Enrichment analysis TAL1', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG24 GO BP Enrichment analysis TAL1')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG24 GO BP Enrichment analysis TAL1', cmap='RdBu_r')

### SG26

In [ ]:
len(deg_tal1_sig_sg26) 

glist_sg26 = deg_tal1_sig_sg26['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_tal1_sg26.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_tal1_sg26.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG26 Enrichment Analysis Results')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG26 KEGG Enrichment analysis TAL1', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG26 GO BP Enrichment analysis TAL1')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG26 GO BP Enrichment analysis TAL1', cmap='RdBu_r')

### SG28

In [ ]:
glist_sg28 = deg_tal1_sig_sg28['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_tal1_sg28.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_tal1_sg28.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG28 Enrichment Analysis Results')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG28 KEGG Enrichment analysis TAL1', cmap='RdBu_r')

#gsplot.barplot(enrichr_results_gobp.res2d, title='SG28 GO BP Enrichment analysis TAL1')
#dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG28 GO BP Enrichment analysis TAL1', cmap='RdBu_r')

## IMM

In [ ]:
# List of dataframe names
imm_names = ['deg_imm_sig_sg24', 'deg_imm_sig_sg26', 'deg_imm_sig_sg28']

### Loop Up/Downregulated KEGG

In [ ]:
# Loop over the dataframes
for df_name in imm_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
    sc.pl.DotPlot(adata_IMM, use_raw=True, var_names = df.names, groupby='sample', standard_scale="var", title=f"{df_short_name} all", cmap='BuGn', figsize=(17, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
    
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    sc.pl.DotPlot(adata_IMM, use_raw=True, var_names = degs_up.names, groupby='sample', standard_scale="var", title=f"{df_short_name} upregulated", cmap='BuGn', figsize=(15, 5)).savefig('./dotplot_up.pdf' ,format='pdf')
    sc.pl.DotPlot(adata_IMM, use_raw=True, var_names = degs_dw.names, groupby='sample', standard_scale="var", title=f"{df_short_name} downregulated", cmap='BuGn', figsize=(15, 5)).savefig('./dotplot_dwn.pdf' ,format='pdf')
   
    # Enrichr API
    # Check if degs_dw contains only one gene and convert it to a list
    if (degs_dw['names'].count() == 1):
        print(df_short_name)
        enr_dw = gp.enrichr(degs_dw,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
    else:
        enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
        
     # Check if degs_dw contains only one gene and convert it to a list
    if (degs_up['names'].count() == 1):
        print(df_short_name)
        enr_up = gp.enrichr(degs_up,
                        gene_sets=['KEGG_2019_Mouse'],
                        outdir=None)
    else:
        enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)   
   
    enr_full = gp.enrichr(df.names,
                        gene_sets=['KEGG_2019_Mouse'],
                        organism='Mouse',
                        outdir=None)

   # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG upregulated", cmap=plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG downregulated", cmap=plt.cm.winter_r, size=5, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        #gp.dotplot(enr_full.res2d, figsize=(3, 5), title=f"{df_short_name} KEGG all", cmap=plt.cm.winter_r, size=5)
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} KEGG all", cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()


    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-KEGG",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-KEGG",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()

### Loop Up/Downregulated GO BP

In [ ]:
# Loop over the dataframes
for df_name in imm_names:
    # Get the dataframe
    df = globals()[df_name]
    
     # Remove prefix from dataframe name
    df_short_name = df_name.replace("deg_", "")
    df_short_name = df_short_name.replace("sig_", "")
    
  
    # Filter the dataframe
    degs_up = df[df.logfoldchanges > 0]
    degs_dw = df[df.logfoldchanges < 0]
    
    # Enrichr API
     # Enrichr API
    # Check if degs_dw contains only one gene and convert it to a list
    if (degs_dw['names'].count() == 1):
        print(df_short_name)
        enr_dw = gp.enrichr(degs_dw,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
    else:
        enr_dw = gp.enrichr(degs_dw.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
        
     # Check if degs_dw contains only one gene and convert it to a list
    if (degs_up['names'].count() == 1):
        print(df_short_name)
        enr_up = gp.enrichr(degs_up,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)
    else:
        enr_up = gp.enrichr(degs_up.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        outdir=None)   
   
    enr_full = gp.enrichr(df.names,
                        gene_sets=['GO_Biological_Process_2021'],
                        organism='Mouse',
                        outdir=None)
    
    
    # trim (go:...)
    enr_up.res2d.Term = enr_up.res2d.Term.str.split(" \(GO").str[0]
    enr_dw.res2d.Term = enr_dw.res2d.Term.str.split(" \(GO").str[0]
    
    # dotplot
    try:
        gp.dotplot(enr_up.res2d, figsize=(3,5), title=f"{df_short_name} GO BP upregulated", cmap = plt.cm.autumn_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for upregulated genes in {df_short_name}: {e}")
    plt.show()

    try:
        gp.dotplot(enr_dw.res2d, figsize=(3,5), title=f"{df_short_name} GO BP downregulated", cmap = plt.cm.winter_r, cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for downregulated genes in {df_short_name}: {e}")
    plt.show()
    
    try:
        gsplot.barplot(enr_full.res2d, title=f"{df_short_name} GO BP all", cutoff = 0.5)
    except ValueError as e:
        print(f"No enrichment terms for all genes in {df_short_name}: {e}")
    plt.show()

    # concat results
    enr_up.res2d['UP_DW'] = "UP"
    enr_dw.res2d['UP_DW'] = "DOWN"
    enr_res = pd.concat([enr_up.res2d.head(), enr_dw.res2d.head()])

    # display multi-datasets
    ax = gp.dotplot(enr_res,figsize=(3,5),
                    x='UP_DW',
                    x_order = ["UP","DOWN"],
                    title=f"{df_short_name}-GO_BP",
                    cmap = NbDr.reversed(),
                    size=3,
                    show_ring=True, cutoff = 0.5)
    ax.set_xlabel("")
    plt.show()

    ax = gp.barplot(enr_res, figsize=(3,5),
                    group ='UP_DW',
                    title =f"{df_short_name}-GO_BP",
                    color = ['b','r'], cutoff = 0.5)
    plt.show()

In [ ]:
glist_sg18 = deg_imm_sig_sg18['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg18,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg18,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_imm_sg18.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_imm_sg18.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG18 Enrichment Analysis Results')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG18 KEGG Enrichment analysis IMM', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG18 GO BP Enrichment analysis IMM')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG18 GO BP Enrichment analysis IMM', cmap='RdBu_r')

### SG24

In [ ]:
glist_sg24 = deg_imm_sig_sg24['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg24,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

geneset_kegg = enrichr_results.results['Genes'] # this column contains the genes for each significant pathway
geneset_gobp = enrichr_results_gobp.results['Genes'] 

gs_kegg_immSG24_list = [genes.split(';') for genes in geneset_kegg]
gs_gobp_immSG24_list = [genes.split(';') for genes in geneset_gobp]

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_imm_sg24.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_imm_sg24.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG24 Enrichment Analysis Results')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG24 KEGG Enrichment analysis IMM', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG24 GO BP Enrichment analysis IMM')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG24 GO BP Enrichment analysis IMM', cmap='RdBu_r')

### SG26

In [ ]:
len(deg_imm_sig_sg26) 

glist_sg26 = deg_imm_sig_sg26['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg26,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

geneset_kegg = enrichr_results.results['Genes'] # this column contains the genes for each significant pathway
geneset_gobp = enrichr_results_gobp.results['Genes'] 

gs_kegg_immSG26_list = [genes.split(';') for genes in geneset_kegg]
gs_gobp_immSG26_list = [genes.split(';') for genes in geneset_gobp]

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_imm_sg26.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_imm_sg26.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG26 Enrichment Analysis Results')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG26 KEGG Enrichment analysis IMM', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG26 GO BP Enrichment analysis IMM')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG26 GO BP Enrichment analysis IMM', cmap='RdBu_r')

### SG28

In [ ]:
len(deg_imm_sig_sg28) 

glist_sg28 = deg_imm_sig_sg28['names'].squeeze().str.strip().tolist()

#Enrichment analysis using enrichr
enrichr_results = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                             gene_sets='KEGG_2019_Mouse',
                            cutoff=0.5)

enrichr_results_gobp = gp.enrichr(gene_list= glist_sg28,
                             organism='Mouse',
                            gene_sets='GO_Biological_Process_2021',
                           # background='mmusculus_gene_ensembl'
                            )

geneset_kegg = enrichr_results.results['Genes'] # this column contains the genes for each significant pathway
geneset_gobp = enrichr_results_gobp.results['Genes'] 

gs_kegg_immSG28_list = [genes.split(';') for genes in geneset_kegg]
gs_gobp_immSG28_list = [genes.split(';') for genes in geneset_gobp]

enrichr_results.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/kegg_imm_sg28.csv')
enrichr_results_gobp.results.to_csv('/data/projects/manuela/scRNA_aging-mouse/enrichment_tables_vsSG20/gobp_imm_sg28.csv')

# Visualize the results
gsplot.barplot(enrichr_results.res2d, title='SG28 Enrichment Analysis Results')
dotplot(enrichr_results.results, cutoff=1, top_term=20, title='SG28 KEGG Enrichment analysis IMM', cmap='RdBu_r')

gsplot.barplot(enrichr_results_gobp.res2d, title='SG28 GO BP Enrichment analysis IMM')
dotplot(enrichr_results_gobp.results, cutoff=1, top_term=20, title='SG28 GO BP Enrichment analysis IMM', cmap='RdBu_r')

In [ ]:
#sc.pl.umap(adata, color='celltype')

### Either PROGENy or DoRothEA model

PROGENy is a comprehensive resource containing a curated collection of pathways and their target genes, with weights for each interaction

In [ ]:
#model = dc.get_progeny(organism = 'Mus musculus')
#model

#model = dc.get_progeny(organism='human', top=100)
#model

import omnipath
from pypath.utils import homology, mapping

progeny = omnipath.requests.Annotations.get(resources = 'PROGENy', wide = True)

#progeny['mouse_uniprot'] = [homology.translate(u, 10090) for u in progeny.uniprot]
#progeny = progeny.explode('mouse_uniprot')
#progeny['mouse_genesymbol'] = [mapping.label(u, ncbi_tax_id = 10090) for u in progeny.mouse_uniprot]

In [ ]:
model = dc.get_dorothea(organism = 'mouse')
model

### Activity inference with Multivariate Linear Model

In [ ]:
dc.run_mlm(mat=adata, net=model, source='source', target='target', weight='weight', verbose=True, min_n=1)

In [ ]:
adata.obs.columns

In [ ]:
acts = dc.get_acts(adata, obsm_key='mlm_estimate')
acts

In [ ]:
sc.pl.umap(acts)
#sc.pl.umap(acts, color='Trail', vcenter=0, cmap='coolwarm')

In [ ]:
mean_acts = dc.summarize_acts(acts, groupby='celltype', min_std=0)
mean_acts

In [ ]:
sns.clustermap(mean_acts, xticklabels=mean_acts.columns, vmin=-2, vmax=2, cmap='coolwarm', row_cluster=False, figsize=(30, 10))
plt.show()

In [ ]:
#dc.show_resources()

In [ ]:
msigdb_mouse = msigdb.msigdb_download_collections(organism = 'mouse')
msigdb_mouse = pd.DataFrame(
    [
        (collname, collcode, gset, gene)
        for (collname, collcode), coll in msigdb_mouse.items()
        for gset, genes in coll.items()
        for gene in genes
    ],
    columns = ['collection', 'code', 'geneset', 'genesymbol']
)

msigdb_mouse

In [ ]:
dc.run_ora(mat=adata, net=msigdb, source='geneset', target='genesymbol', verbose=True)

In [ ]:
acts = dc.get_acts(adata, obsm_key='ora_estimate')
acts

In [ ]:
adata.obsm['ora_estimate']

In [ ]:
sc.pl.umap(acts, color='REACTOME_INTRINSIC_PATHWAY_OF_FIBRIN_CLOT_FORMATION')

In [ ]:
mean_enr = dc.summarize_acts(acts, groupby='celltype', min_std=0)
mean_enr

In [ ]:
sns.clustermap(mean_enr, xticklabels=mean_enr.columns, vmax=20, cmap='viridis')
plt.show()

In [ ]:
dc.run_ora(mat=adata, net=msigdb, source='geneset', target='genesymbol', verbose=True)

In [ ]:
acts = dc.get_acts(adata, obsm_key='ora_estimate')

In [ ]:
absolute_sums = np.abs(adata.obsm['ora_estimate']).sum(axis=0)
sorted_indices = np.argsort(absolute_sums)[::-1]  # Sort indices in descending order
sorted_indices = np.argsort(absolute_sums)[::-1]  # Sort indices in descending order

top_n = 10  
top_pathways = acts[:, sorted_indices[:top_n]]
sorted_indices.index

In [ ]:
for i, pathway in enumerate(sorted_indices.index):
    sc.pl.umap(acts, color=pathway)

In [ ]:
for i, pathway in enumerate(top_pathways.columns):
    adata.obs[pathway] = top_pathways[:, i]


In [ ]:
# Assuming `top_pathways.columns` contains the names of the top pathways
sc.pl.umap(adata, color=top_pathways.columns, legend_loc='on data')


# GSEA

# Close pdfs

In [ ]:
pdf_pages.close()
pdf_dotplots.close()